# FER-YOLO-Mamba - RunPod Training
Run cells top to bottom. Each cell prints clear status.

In [ ]:
# CELL 1 - GPU CHECK
import torch, os, sys, subprocess
assert torch.cuda.is_available(), 'No GPU found!'
p = torch.cuda.get_device_properties(0)
print(f'GPU   : {p.name}')
print(f'VRAM  : {p.total_memory/1e9:.1f} GB')
print(f'Torch : {torch.__version__}')
print(f'CUDA  : {torch.version.cuda}')

In [ ]:
# CELL 2 - CLONE / UPDATE CODE
REPO    = 'https://github.com/satyamsingh5512/FER-YOLO.git'
WORKDIR = '/workspace/FER-YOLO'

if os.path.exists(WORKDIR):
    r = subprocess.run(['git', 'pull'], cwd=WORKDIR, capture_output=True, text=True)
    print(r.stdout.strip())
else:
    r = subprocess.run(['git', 'clone', REPO, WORKDIR], capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr)
    print('Cloned.')

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Dir:', os.getcwd())

In [ ]:
# CELL 3 - INSTALL DEPENDENCIES (~10 min for mamba-ssm)
for _p in ['/usr/local/cuda', '/usr/local/cuda-12', '/usr/local/cuda-12.9', '/usr/local/cuda-12.4']:
    if os.path.isdir(_p):
        os.environ['CUDA_HOME'] = _p
        os.environ['PATH'] = os.path.join(_p, 'bin') + ':' + os.environ.get('PATH', '')
        print('CUDA_HOME =', _p)
        break
os.environ['MAX_JOBS'] = '4'

def sh(cmd):
    r = subprocess.run(cmd)
    return r.returncode == 0

sh([sys.executable, '-m', 'pip', 'install', '-q', 'packaging', 'einops', 'timm>=0.6.13,<0.9.0', 'gdown'])
print('packaging / einops / timm / gdown - OK')

print('Installing mamba-ssm (~10 min)...')
ok = (sh([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', 'mamba-ssm>=2.0.3'])
   or sh([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', 'mamba-ssm>=2.0.3']))
if not ok:
    raise RuntimeError('mamba-ssm install failed. Check log above.')

import selective_scan_cuda
print('selective_scan_cuda - OK')
print('All dependencies ready.')

In [ ]:
# CELL 4 - GET DATASET
# The dataset is ImageFolder format:
#   train/angry/*.jpg  train/happy/*.jpg  etc.
# Upload your zip to /workspace OR share the Google Drive file
# (Share -> Anyone with the link -> Viewer) and set GDRIVE_ID below.
import glob, zipfile

GDRIVE_ID  = '1C9-WD-e4ojpZo9i6IxrG5lgiJY6RRYQk'  # your Google Drive file ID
DATA_DIR   = '/workspace/dataset'
ZIP_PATH   = os.path.join(DATA_DIR, 'fer2013.zip')
os.makedirs(DATA_DIR, exist_ok=True)

# skip download if already extracted
already_extracted = any(
    os.path.isdir(os.path.join(DATA_DIR, d))
    for d in ['split_FER2013_original', 'train']
)

if not already_extracted:
    existing_zip = glob.glob('/workspace/**/*fer*.zip', recursive=True) + \
                   glob.glob('/workspace/**/*split*.zip', recursive=True)
    if existing_zip:
        ZIP_PATH = existing_zip[0]
        print('Found existing zip:', ZIP_PATH)
    elif not os.path.exists(ZIP_PATH):
        import gdown
        print('Downloading from Google Drive...')
        gdown.download(id=GDRIVE_ID, output=ZIP_PATH, quiet=False)

    if os.path.exists(ZIP_PATH):
        print('Extracting...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(DATA_DIR)
        print('Done.')
else:
    print('Dataset already extracted.')

# find the root folder that has train/ + val/ (or test/)
DATASET_ROOT = None
for base in [DATA_DIR, '/workspace']:
    for root, dirs, _ in os.walk(base):
        if root.replace(base, '').count(os.sep) > 4:
            dirs[:] = []
            continue
        if 'train' in dirs and ('val' in dirs or 'test' in dirs):
            DATASET_ROOT = root
            break
    if DATASET_ROOT: break

if not DATASET_ROOT:
    print('/workspace contents:', os.listdir('/workspace'))
    raise RuntimeError('Dataset not found. Set DATASET_ROOT manually.')

print('DATASET_ROOT:', DATASET_ROOT)
print('Contents    :', sorted(os.listdir(DATASET_ROOT)))

In [ ]:
# CELL 5 - GENERATE ANNOTATION FILES
# Converts ImageFolder layout -> repo annotation format.
# Force-reloads module so git pull changes are always picked up.
import importlib, sys as _sys
if 'convert_yolo_to_annotation' in _sys.modules:
    importlib.reload(_sys.modules['convert_yolo_to_annotation'])
from convert_yolo_to_annotation import convert

# remove stale files from previous runs
for f in ['raf_train.txt', 'raf_val.txt']:
    p = os.path.join(WORKDIR, f)
    if os.path.exists(p): os.remove(p)

convert(DATASET_ROOT, WORKDIR)

print()
for f in ['raf_train.txt', 'raf_val.txt']:
    lines = open(os.path.join(WORKDIR, f)).readlines()
    print(f'{f}: {len(lines)} samples')
    if lines:
        print('  sample ->', lines[0].strip()[:90])
    else:
        raise RuntimeError(f'{f} is empty! Check Cell 4 output.')

In [ ]:
# CELL 6 - PRE-FLIGHT CHECK
import collections
cls_ids = set()
for fn in ['raf_train.txt', 'raf_val.txt']:
    for line in open(fn):
        for box in line.strip().split()[1:]:
            cls_ids.add(int(box.split(',')[4]))
n_classes = (max(cls_ids) + 1) if cls_ids else 0
print(f'Classes detected : {sorted(cls_ids)}  ->  num_classes = {n_classes}')

classes_path = 'model_data/fer_classes.txt'
os.environ['CLASSES_PATH'] = classes_path
print('CLASSES_PATH =', classes_path)

from nets.yolo import YoloBody
_ = YoloBody(n_classes, 's'); del _
print('YoloBody init - OK')

import selective_scan_cuda
print('selective_scan_cuda - OK')

for f in ['train_kaggle.py', 'raf_train.txt', 'raf_val.txt', classes_path]:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'{f} - OK')

print('\nReady to train.')
print(f'Config: 224x224, batch=32, epochs=250, patience=50, fp16=False, device=cuda:0')

In [ ]:
# CELL 7 - TRAIN
# Settings: 224x224, batch=32, epochs=250, patience=50, fp16=False
# All set in train_kaggle.py - no changes needed here.
env = dict(os.environ)
proc = subprocess.Popen(
    [sys.executable, '-u', 'train_kaggle.py'],
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Training failed (code {proc.returncode}). Scroll up for the error.')
print('\nTraining finished.')

In [ ]:
# CELL 8 - RESULTS
import glob
ckpts = sorted(glob.glob('logs/**/*.pth', recursive=True))
print(f'Checkpoints ({len(ckpts)}):')
for c in ckpts:
    tag = ' <- BEST' if 'best' in c else (' <- LAST' if 'last' in c else '')
    print(f'  {c}  ({os.path.getsize(c)/1e6:.1f} MB){tag}')
print('\nDownload logs/ from the RunPod file browser.')